In [1]:
import csv 
import json
import os
import pandas as pd
import random
from pathlib import Path
import warnings
import re

In [2]:
# Silence openpyxl formatting warnings (safe)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

path_severity_folder = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity"

output_file = os.path.join(
    path_severity_folder,
    "Merged_Core_Indicators_All_Months_FIXED.xlsx"
)

excel_files = [
    f for f in os.listdir(path_severity_folder)
    if f.lower().endswith((".xlsx", ".xls")) and not f.startswith("~$")
]

MONTH_PATTERN = (
    r"(january|february|march|april|may|june|"
    r"july|august|september|october|november|december)"
)
YEAR_PATTERN = r"(20\d{2})"

def extract_month_year(filename):
    """
    Extracts the month and the year that comes **after the month** in a filename.
    Ignores extra underscores or trailing numbers after the year.
    """
    name = filename.lower()

    # Find the month
    month_match = re.search(MONTH_PATTERN, name)
    if not month_match:
        raise ValueError(f"Month not found in filename: {filename}")
    month = month_match.group(1).capitalize()

    # Only look for the year **after the month**
    rest_of_name = name[month_match.end():]  # slice after month
    # Match a year possibly followed by underscores and digits, e.g., "_2021_0"
    year_match = re.search(r"20\d{2}", rest_of_name)
    if not year_match:
        raise ValueError(f"Year not found after month in filename: {filename}")
    year = year_match.group(0)

    return month, year

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for file in sorted(excel_files):
        file_path = os.path.join(path_severity_folder, file)

        try:
            month, year = extract_month_year(file)
            sheet_name = f"CoreIndicators_{month[:3]}_{year}"

            df = pd.read_excel(file_path, sheet_name="Core Indicators")
            df.to_excel(writer, sheet_name=sheet_name, index=False)

            print(f"Added: {sheet_name}")

        except Exception as e:
            print(f"Skipped {file} → {e}")

print("\n✅ Finished")
print(f"Created:\n{output_file}")

Added: CoreIndicators_Sep_2020
Added: CoreIndicators_Oct_2020
Added: CoreIndicators_Nov_2020
Added: CoreIndicators_Dec_2020
Added: CoreIndicators_Jan_2021
Added: CoreIndicators_Feb_2021
Added: CoreIndicators_Mar_2021
Added: CoreIndicators_Apr_2021
Added: CoreIndicators_May_2021
Added: CoreIndicators_Jun_2021
Added: CoreIndicators_Jul_2021
Added: CoreIndicators_Aug_2021
Added: CoreIndicators_Sep_2021
Added: CoreIndicators_Oct_2021
Added: CoreIndicators_Nov_2021
Added: CoreIndicators_Dec_2021
Added: CoreIndicators_Jan_2022
Added: CoreIndicators_Feb_2022
Added: CoreIndicators_Mar_2022
Added: CoreIndicators_Apr_2022
Added: CoreIndicators_May_2022
Added: CoreIndicators_Jun_2022
Added: CoreIndicators_Jul_2022
Added: CoreIndicators_Aug_2022
Added: CoreIndicators_Sep_2022
Added: CoreIndicators_Oct_2022
Added: CoreIndicators_Nov_2022
Added: CoreIndicators_Dec_2022
Added: CoreIndicators_Jan_2023
Added: CoreIndicators_Feb_2023
Added: CoreIndicators_Mar_2023
Added: CoreIndicators_Apr_2023
Added: C

In [3]:
file_path = '/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_All_Months_FIXED.xlsx'

# Load the Excel file and get sheet names
excel_obj = pd.ExcelFile(file_path)
sheet_names = excel_obj.sheet_names

print("Sheets inside the file:")
for sheet in sheet_names:
    print(f"- {sheet}")

Sheets inside the file:
- CoreIndicators_Sep_2020
- CoreIndicators_Oct_2020
- CoreIndicators_Nov_2020
- CoreIndicators_Dec_2020
- CoreIndicators_Jan_2021
- CoreIndicators_Feb_2021
- CoreIndicators_Mar_2021
- CoreIndicators_Apr_2021
- CoreIndicators_May_2021
- CoreIndicators_Jun_2021
- CoreIndicators_Jul_2021
- CoreIndicators_Aug_2021
- CoreIndicators_Sep_2021
- CoreIndicators_Oct_2021
- CoreIndicators_Nov_2021
- CoreIndicators_Dec_2021
- CoreIndicators_Jan_2022
- CoreIndicators_Feb_2022
- CoreIndicators_Mar_2022
- CoreIndicators_Apr_2022
- CoreIndicators_May_2022
- CoreIndicators_Jun_2022
- CoreIndicators_Jul_2022
- CoreIndicators_Aug_2022
- CoreIndicators_Sep_2022
- CoreIndicators_Oct_2022
- CoreIndicators_Nov_2022
- CoreIndicators_Dec_2022
- CoreIndicators_Jan_2023
- CoreIndicators_Feb_2023
- CoreIndicators_Mar_2023
- CoreIndicators_Apr_2023
- CoreIndicators_May_2023
- CoreIndicators_Jun_2023
- CoreIndicators_Jul_2023
- CoreIndicators_Aug_2023
- CoreIndicators_Sep_2023
- CoreIndicato

In [4]:
# Input and output files
input_file = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_All_Months_FIXED.xlsx"
output_file = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Drivers.xlsx"

# Crisis labels to keep (case-insensitive, partial match)
KEYWORDS = [
    "socio-political",
    "violence",
    "conflict",
    "displacement",
    "international displacement",
    "multiple crises country",
    "regional crisis",
    "political and economic crisis",
]

pattern = re.compile("|".join(KEYWORDS), re.IGNORECASE)

xls = pd.ExcelFile(input_file)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet)

        # Determine which column to use
        if "Drivers" in df.columns:
            col = "Drivers"
        elif "Type of crisis" in df.columns:
            col = "Type of crisis"
        else:
            print(f"Skipped {sheet} → no Drivers / Type of crisis column")
            continue

        # Filter rows
        filtered_df = df[
            df[col]
            .astype(str)
            .str.contains(pattern, na=False)
        ]

        # Save filtered data
        filtered_df.to_excel(writer, sheet_name=sheet, index=False)

        print(f"{sheet}: {len(filtered_df)} rows kept (using '{col}')")

print("\n✅ Filtering complete")
print(f"Filtered file saved as:\n{output_file}")

CoreIndicators_Sep_2020: 88 rows kept (using 'Type of crisis')
CoreIndicators_Oct_2020: 91 rows kept (using 'Type of crisis')
CoreIndicators_Nov_2020: 92 rows kept (using 'Type of crisis')
CoreIndicators_Dec_2020: 94 rows kept (using 'Type of crisis')
CoreIndicators_Jan_2021: 94 rows kept (using 'Type of crisis')
CoreIndicators_Feb_2021: 92 rows kept (using 'Type of crisis')
CoreIndicators_Mar_2021: 93 rows kept (using 'Type of crisis')
CoreIndicators_Apr_2021: 95 rows kept (using 'Type of crisis')
CoreIndicators_May_2021: 93 rows kept (using 'Type of crisis')
CoreIndicators_Jun_2021: 93 rows kept (using 'Type of crisis')
CoreIndicators_Jul_2021: 93 rows kept (using 'Type of crisis')
CoreIndicators_Aug_2021: 93 rows kept (using 'Type of crisis')
CoreIndicators_Sep_2021: 94 rows kept (using 'Type of crisis')
CoreIndicators_Oct_2021: 94 rows kept (using 'Type of crisis')
CoreIndicators_Nov_2021: 94 rows kept (using 'Type of crisis')
CoreIndicators_Dec_2021: 95 rows kept (using 'Type of c

In [5]:
# Input and output files
input_file = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Drivers.xlsx"
output_file = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Reliability.xlsx"

xls = pd.ExcelFile(input_file)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet)

        # Find the position of 'people in need'
        if "People in Need" not in df.columns:
            print(f"Skipped {sheet} → no 'people in need' column")
            continue

        start_idx = df.columns.get_loc("People in Need")
        reliability_idx = start_idx + 4  # 4 columns after

        if reliability_idx >= len(df.columns):
            print(f"Skipped {sheet} → reliability column out of bounds")
            continue

        reliability_col = df.columns[reliability_idx]

        # Filter rows where reliability is 1 or 2
        filtered_df = df[df[reliability_col].isin([1, 2])]

        # Save filtered data
        filtered_df.to_excel(writer, sheet_name=sheet, index=False)

        print(f"{sheet}: {len(filtered_df)} rows kept (filtered by '{reliability_col}')")

print("\n✅ Second filtering complete")
print(f"Filtered file saved as:\n{output_file}")

CoreIndicators_Sep_2020: 53 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Oct_2020: 53 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Nov_2020: 54 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Dec_2020: 56 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Jan_2021: 53 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Feb_2021: 47 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Mar_2021: 47 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Apr_2021: 50 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_May_2021: 47 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Jun_2021: 49 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Jul_2021: 52 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Aug_2021: 50 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Sep_2021: 53 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Oct_2021: 54 rows kept (filtered by 'Unnamed: 105')
CoreIndicators_Nov_2021: 57 rows kept (filtered by 'Unnamed: 1

In [6]:
# Input and output files
input_file = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Reliability.xlsx"
output_file = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Sources.xlsx"

# Regex pattern to match dates with day/month/year or month/year
# Matches:
# 16/06/2021, 16-06-2021, 02/2021, 02-2021
# Does NOT match just "2021"
date_pattern = re.compile(r'(\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b)|(\b\d{1,2}[/-]\d{2,4}\b)')

xls = pd.ExcelFile(input_file)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet)

        # Find the position of 'people in need'
        if "People in Need" not in df.columns:
            print(f"Skipped {sheet} → no 'people in need' column")
            continue

        start_idx = df.columns.get_loc("People in Need")
        source_idx = start_idx + 2  # 2 columns after 'people in need'

        if source_idx >= len(df.columns):
            print(f"Skipped {sheet} → source column out of bounds")
            continue

        source_col = df.columns[source_idx]

        # Keep rows that contain at least one full date
        filtered_df = df[df[source_col].astype(str).str.contains(date_pattern)]

        # Save filtered data
        filtered_df.to_excel(writer, sheet_name=sheet, index=False)

        print(f"{sheet}: {len(filtered_df)} rows kept (filtered by '{source_col}')")

print("\n✅ Source date filtering complete")
print(f"Filtered file saved as:\n{output_file}")

/tmp/ipykernel_1388/3402233542.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_df = df[df[source_col].astype(str).str.contains(date_pattern)]


CoreIndicators_Sep_2020: 45 rows kept (filtered by 'Unnamed: 103')


/tmp/ipykernel_1388/3402233542.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_df = df[df[source_col].astype(str).str.contains(date_pattern)]


CoreIndicators_Oct_2020: 44 rows kept (filtered by 'Unnamed: 103')


/tmp/ipykernel_1388/3402233542.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_df = df[df[source_col].astype(str).str.contains(date_pattern)]


...

CoreIndicators_Nov_2025: 34 rows kept (filtered by 'Unnamed: 103')


/tmp/ipykernel_1388/3402233542.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_df = df[df[source_col].astype(str).str.contains(date_pattern)]


CoreIndicators_Dec_2025: 34 rows kept (filtered by 'Unnamed: 103')


/tmp/ipykernel_1388/3402233542.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_df = df[df[source_col].astype(str).str.contains(date_pattern)]


CoreIndicators_Jan_2026: 35 rows kept (filtered by 'Unnamed: 103')

✅ Source date filtering complete
Filtered file saved as:
/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Sources.xlsx


In [7]:
from datetime import datetime, timedelta
import calendar

# Input and output files
input_file = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Sources.xlsx"
output_file = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Sheet_Dates.xlsx"

# Regex to extract dates: day/month/year or month/year
date_pattern = re.compile(r'\b(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})\b|\b(\d{1,2})[/-](\d{2,4})\b')

# Map month names to numbers
month_map = {m.lower(): i for i, m in enumerate(calendar.month_abbr) if m}

def parse_sheet_date(sheet_name):
    """Extract month and year from sheet name like 'CoreIndicators_Dec_2021'"""
    parts = sheet_name.split('_')
    for part in parts:
        if part.lower() in month_map:
            month = month_map[part.lower()]
        elif part.isdigit() and len(part) == 4:
            year = int(part)
    return month, year

def extract_dates(cell):
    """Return list of datetime objects from cell string"""
    dates = []
    cell = str(cell)
    for match in date_pattern.finditer(cell):
        if match.group(1):  # full date d/m/y
            day = int(match.group(1))
            month = int(match.group(2))
            year = int(match.group(3))
        else:  # m/y
            day = 1
            month = int(match.group(4))
            year = int(match.group(5))
        if year < 100:  # two-digit year
            year += 2000
        try:
            dates.append(datetime(year, month, day))
        except ValueError:
            continue
    return dates

xls = pd.ExcelFile(input_file)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    number_events = 0
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet)

        # Find column indices
        if "People in Need" not in df.columns:
            print(f"Skipped {sheet} → no 'People in Need' column")
            continue

        start_idx = df.columns.get_loc("People in Need")
        source_idx = start_idx + 2  # source dates column

        if source_idx >= len(df.columns):
            print(f"Skipped {sheet} → source column out of bounds")
            continue

        source_col = df.columns[source_idx]

        # Get month/year of sheet
        try:
            month, year = parse_sheet_date(sheet)
        except Exception:
            print(f"Skipped {sheet} → cannot parse month/year from sheet name")
            continue

        # Compute end of the current month
        last_day = calendar.monthrange(year, month)[1]
        end_range = datetime(year, month, last_day)

        # Compute start date 3 months back
        if month > 2:
            # Simple case: subtract 3 months within the same year
            start_month = month - 2
            start_year = year
        else:
            # Need to wrap around to previous year
            start_month = 12 - (2 - month)
            start_year = year - 1

        start_range = datetime(start_year, start_month, 1)

        # Filter rows: keep if at least one date in range
        def has_date_in_range(cell):
            for dt in extract_dates(cell):
                if start_range <= dt <= end_range:
                    return True
            return False

        filtered_df = df[df[source_col].apply(has_date_in_range)]

        # Save filtered sheet
        filtered_df.to_excel(writer, sheet_name=sheet, index=False)

        print(f"{sheet}: {len(filtered_df)} rows kept based on source dates")
        
        number_events = number_events + len(filtered_df)

print("\n✅ Sheet date filtering complete")
print(f"Filtered file saved as:\n{output_file}")
print('total number of resulting events: ', number_events)

CoreIndicators_Sep_2020: 19 rows kept based on source dates
CoreIndicators_Oct_2020: 17 rows kept based on source dates
CoreIndicators_Nov_2020: 20 rows kept based on source dates
CoreIndicators_Dec_2020: 15 rows kept based on source dates
CoreIndicators_Jan_2021: 17 rows kept based on source dates
CoreIndicators_Feb_2021: 19 rows kept based on source dates
CoreIndicators_Mar_2021: 15 rows kept based on source dates
CoreIndicators_Apr_2021: 15 rows kept based on source dates
CoreIndicators_May_2021: 11 rows kept based on source dates
CoreIndicators_Jun_2021: 15 rows kept based on source dates
CoreIndicators_Jul_2021: 20 rows kept based on source dates
CoreIndicators_Aug_2021: 16 rows kept based on source dates
CoreIndicators_Sep_2021: 18 rows kept based on source dates
CoreIndicators_Oct_2021: 17 rows kept based on source dates
CoreIndicators_Nov_2021: 20 rows kept based on source dates
CoreIndicators_Dec_2021: 22 rows kept based on source dates
CoreIndicators_Jan_2022: 23 rows kept ba

In [8]:
# Input Excel file and output CSV
file_path = '/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Filtered_Sheet_Dates.xlsx'
output_csv = '/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Stacked.csv'

# Read the Excel file
xls = pd.ExcelFile(file_path)

all_rows = []

for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet)
    
    # Ensure necessary columns exist
    if 'ISO3 code' not in df.columns or 'Crisis ID' not in df.columns:
        print(f"Skipped sheet {sheet} → missing 'ISO3 code' or 'Crisis ID'")
        continue

    # Handle rows with multiple countries separated by commas or semicolons
    for idx, row in df.iterrows():
        countries = str(row['ISO3 code']).split(',')  # split by comma
        if len(countries) == 1:
            countries = str(row['ISO3 code']).split(';')  # try splitting by semicolon

        for country in countries:
            new_row = row.copy()
            new_row['ISO3 code'] = country.strip()
            all_rows.append(new_row)

# Combine everything into a single DataFrame
stacked_df = pd.DataFrame(all_rows)

# Optional: sort by country, then by Crisis ID
stacked_df.sort_values(by=['ISO3 code', 'Crisis ID'], inplace=True)

# Save to CSV
stacked_df.to_csv(output_csv, index=False)

print(f"\n✅ All sheets stacked and saved to:\n{output_csv}")


✅ All sheets stacked and saved to:
/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Stacked.csv


In [9]:
import pandas as pd
from openpyxl import load_workbook


excel_path  = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/20220804_inform_severity_-_july_2022.xlsx"
csv_path  = '/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Stacked.csv'

# ---- read CSV headers ----
csv_headers = pd.read_csv(csv_path, nrows=0).columns

# Load workbook and select the correct sheet
wb = load_workbook(excel_path, data_only=True)
ws = wb["Core Indicators"]

# Infer true column count from the sheet
max_col = 0
for row in ws.iter_rows():
    for cell in row:
        if cell.value is not None:
            max_col = max(max_col, cell.column)

# Explicitly read row 1 and 2 from that sheet
row1 = [ws.cell(row=1, column=c).value for c in range(1, max_col + 1)]
row2 = [ws.cell(row=2, column=c).value for c in range(1, max_col + 1)]

row1 = pd.Series(row1)
row2 = pd.Series(row2)

# ---- force print first 30 entries ----
print("=== Excel Row 1 (first 30 columns) ===")
print(row1.iloc[:30])

print("\n=== Excel Row 2 (first 30 columns) ===")
print(row2.iloc[:30])

print("\n=== CSV Headers (first 30 columns) ===")
print(list(csv_headers[:30]))

=== Excel Row 1 (first 30 columns) ===
0                Crisis
1        Type of crisis
2             Crisis ID
3               Country
4             ISO3 code
5      Total population
6                  None
7                  None
8                  None
9                  None
10                 None
11                 None
12                 None
13    Landmass affected
14                 None
15                 None
16                 None
17                 None
18                 None
19                 None
20                 None
21       People exposed
22                 None
23                 None
24                 None
25                 None
26                 None
27                 None
28                 None
29      People affected
dtype: object

=== Excel Row 2 (first 30 columns) ===
0                     0
1                  None
2                     0
3                  None
4                     0
5              Helper 0
6                Figure
7                So

In [10]:
# ---- forward-fill row 1 to replace merged cells ----
row1_filled = row1.ffill()

# ---- build new flattened CSV headers ----
new_csv_headers = []

for i in range(len(row1_filled)):
    category = row1_filled.iloc[i]
    subcategory = row2.iloc[i]

    # Clean up None / nan
    if pd.isna(category):
        category = ""
    if pd.isna(subcategory):
        subcategory = ""

    # Build header
    if category and subcategory:
        header = f"{category} | {subcategory}"
    elif category:
        header = str(category)
    elif subcategory:
        header = str(subcategory)
    else:
        header = f"Column_{i}"

    new_csv_headers.append(header)

new_csv_headers = pd.Index(new_csv_headers)

# ---- print first 50 new headers ----
print("\n=== New CSV Headers (first 50 columns) ===")
for i, h in enumerate(new_csv_headers[:50]):
    print(f"{i}: {h}")


=== New CSV Headers (first 50 columns) ===
0: Crisis
1: Type of crisis
2: Crisis ID
3: Country
4: ISO3 code
5: Total population | Helper 0
6: Total population | Figure
7: Total population | Source
8: Total population | Reliability
9: Total population | Reliability Score
10: Total population | Justification
11: Total population | Link
12: Total population | Date
13: Landmass affected | Helper 1
14: Landmass affected | Figure
15: Landmass affected | Source
16: Landmass affected | Reliability
17: Landmass affected | Reliability Score
18: Landmass affected | Justification
19: Landmass affected | Link
20: Landmass affected | Date
21: People exposed | Helper 2
22: People exposed | Figure
23: People exposed | Source
24: People exposed | Reliability
25: People exposed | Reliability Score
26: People exposed | Justification
27: People exposed | Link
28: People exposed | Date
29: People affected | Helper 3
30: People affected | Figure
31: People affected | Source
32: People affected | Reliabilit

In [11]:

print("\n=== CSV headers (first 2) ===")
for i, h in enumerate(df.columns[:2]):
    print(f"{i}: {h}")

print("\n=== CSV headers (last 2) ===")
for i, h in enumerate(df.columns[-2:], start=len(df.columns) - 2):
    print(f"{i}: {h}")

print("\n=== New headers from Excel (first 2) ===")
for i, h in enumerate(new_csv_headers[:2]):
    print(f"{i}: {h}")

print("\n=== New headers from Excel (last 2) ===")
for i, h in enumerate(new_csv_headers[-2:], start=len(new_csv_headers) - 2):
    print(f"{i}: {h}")

print("\n=== Lengths ===")
print("CSV columns:", df.shape[1])
print("Excel-derived headers:", len(new_csv_headers))


=== CSV headers (first 2) ===
0: Crisis
1: Drivers

=== CSV headers (last 2) ===
251: Unnamed: 251
252: Unnamed: 252

=== New headers from Excel (first 2) ===
0: Crisis
1: Type of crisis

=== New headers from Excel (last 2) ===
251: Physical constraints in the environment (obstacles related to terrain, climate, lack of infrastructure, etc.) | Link
252: Physical constraints in the environment (obstacles related to terrain, climate, lack of infrastructure, etc.) | Date

=== Lengths ===
CSV columns: 253
Excel-derived headers: 253


In [12]:
import pandas as pd
from openpyxl import load_workbook
import os

# ---- paths ----
excel_path  = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/20220804_inform_severity_-_july_2022.xlsx"
csv_path  = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Merged_Core_Indicators_Stacked.csv"

# ---- read full CSV ----
df = pd.read_csv(csv_path)

# ---- column to cut at ----
cut_column_name = "People facing limited access constraints"

if cut_column_name not in df.columns:
    raise ValueError(f"Column '{cut_column_name}' not found in CSV headers!")

cut_index = df.columns.get_loc(cut_column_name)
df = df.iloc[:, :cut_index]  # include the cut column

# ---- load Excel workbook & sheet ----
wb = load_workbook(excel_path, data_only=True)
ws = wb["Core Indicators"]

# ---- read row 1 & row 2 from Excel ----
row1 = [ws.cell(row=1, column=c).value for c in range(1, ws.max_column + 1)]
row2 = [ws.cell(row=2, column=c).value for c in range(1, ws.max_column + 1)]

row1 = pd.Series(row1)
row2 = pd.Series(row2)

# forward-fill row1 for merged cells
row1_filled = row1.ffill()

# truncate Excel rows to match truncated CSV
row1_filled = row1_filled.iloc[:len(df.columns)]
row2 = row2.iloc[:len(df.columns)]

# ---- build new flattened headers ----
new_csv_headers = []
for i in range(len(row1_filled)):
    category = row1_filled.iloc[i]
    subcategory = row2.iloc[i]

    if pd.isna(category):
        category = ""
    if pd.isna(subcategory):
        subcategory = ""

    if category and subcategory:
        header = f"{category} | {subcategory}"
    elif category:
        header = str(category)
    elif subcategory:
        header = str(subcategory)
    else:
        header = f"Column_{i}"

    new_csv_headers.append(header)

# ---- apply new headers ----
df.columns = new_csv_headers

# ---- print first and last 5 headers ----
print("=== First 5 CSV headers ===")
for i, h in enumerate(df.columns[:5]):
    print(f"{i}: {h}")

print("\n=== Last 5 CSV headers ===")
for i, h in enumerate(df.columns[-5:], start=len(df.columns)-5):
    print(f"{i}: {h}")

# ---- save new CSV ----
output_csv_path = os.path.join(os.path.dirname(csv_path), "Filtered data.csv")
df.to_csv(output_csv_path, index=False)

print(f"\n✅ Filtered CSV saved to:\n{output_csv_path}")


=== First 5 CSV headers ===
0: Crisis
1: Type of crisis
2: Crisis ID
3: Country
4: ISO3 code

=== Last 5 CSV headers ===
160: Crisis affected groups | Reliability
161: Crisis affected groups | Reliability Score
162: Crisis affected groups | Justification
163: Crisis affected groups | Link
164: Crisis affected groups | Date

✅ Filtered CSV saved to:
/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Filtered data.csv


In [13]:
import pandas as pd
import json

# ---- load cleaned CSV ----
csv_path_cleaned = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Filtered data.csv"
df = pd.read_csv(csv_path_cleaned)

# ---- parse CSV headers to get categories and subfields ----
category_fields = {}
for col in df.columns:
    if '|' in col:
        category, field = [x.strip() for x in col.split('|', 1)]
        if category not in category_fields:
            category_fields[category] = []
        # skip 'Helper' columns
        if 'Helper' not in field:
            category_fields[category].append(field)

# ---- initialize JSON container ----
json_data = {}

# ---- iterate over CSV rows ----
for _, row in df.iterrows():
    country_key = f"{row['ISO3 code']}"
    if country_key not in json_data:
        json_data[country_key] = {}

    crisis_id = str(row['Crisis ID'])
    if crisis_id not in json_data[country_key]:
        json_data[country_key][crisis_id] = {}
        # add type of crisis
        json_data[country_key][crisis_id]["type_of_crisis"] = row["Type of crisis"]

    crisis_dict = json_data[country_key][crisis_id]

    # ---- iterate categories ----
    for category, fields in category_fields.items():
        # prepare current entry, convert NaN -> None
        entry = {}
        for field in fields:
            col_name = f"{category} | {field}"
            if col_name in df.columns:
                value = row[col_name]
                if pd.isna(value):
                    value = None
                entry[field] = value

        # initialize list for this category if not exists
        if category not in crisis_dict:
            crisis_dict[category] = []

        # check last date
        last_entry_date = crisis_dict[category][-1]["Date"] if crisis_dict[category] else None
        current_date = entry.get("Date")

        # add new entry if date changed or list empty
        if current_date != last_entry_date:
            crisis_dict[category].append(entry)


def nan_to_none(obj):
    if isinstance(obj, dict):
        return {k: nan_to_none(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [nan_to_none(x) for x in obj]
    elif isinstance(obj, float) and pd.isna(obj):
        return None
    else:
        return obj

json_data = nan_to_none(json_data)            
            
output_json_path = "/eos/jeodpp/home/users/mihadar/data/INFORM Severity/Filtered_events.json"
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4)

print(f"✅ JSON saved to: {output_json_path}")

✅ JSON saved to: /eos/jeodpp/home/users/mihadar/data/INFORM Severity/Filtered_events.json
